tokenization


# Day 4: Tokenization and Conversation Memory

This notebook covers basic tokenization with `tiktoken` and a simple local Ollama-based chat flow.  
We also inspect "illusion of memory" by maintaining message history in a conversation thread.  

Block 12 corresponds to the conversational request where a potential Ollama/config issue can surface (model not running, missing model, etc.).

In [3]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
print(encoding)
tokens = encoding.encode("she eats deliciouslifized buttertoffee")

<Encoding 'o200k_base'>


## Tokenization with tiktoken

`tiktoken` is an encoding library for OpenAI models.  
Here we encode a test string, decode each token id, and inspect how text is chunked.  
This is useful for understanding prompt length limits and control over token usage.

In [4]:
print(tokens)

[45842, 88971, 19134, 83008, 2110, 18911, 935, 76914]


In [5]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

45842 = she
88971 =  eats
19134 =  delicious
83008 = lif
2110 = ized
18911 =  butter
935 = to
76914 = ffee


In [7]:
encoding.decode([65])

'b'

illusion of memory


## Local Ollama Chat and "Illusion of Memory"

This section uses a persistent `messages` list to simulate conversational context.  
As more turns are added, the model sees past user and assistant messages, creating a sense of memory.  

Block 12 is the first token of the Ollama chat flow where we should validate the model connection and ensure that `ollama run llama3.2` is active.

### What is Ollama?

**Ollama** is a lightweight framework for running Large Language Models (LLMs) locally on your machine. 

Key benefits:
- **No API keys needed**: Run models privately on your hardware
- **Free**: No cloud costs or usage limits
- **OpenAI-compatible API**: Use the same `openai` library with a local endpoint
- **Offline capable**: Works without internet after models are downloaded
- **Fast**: Lower latency than cloud APIs for small requests

Ollama serves models via HTTP at `http://localhost:11434` and provides an OpenAI-compatible `/v1` endpoint, allowing us to swap cloud LLMs for local ones with minimal code changes.

In [20]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from openai import OpenAI

### Setting up the Ollama Client

We import the OpenAI library even though we're using Ollama. This works because:
1. Ollama exposes an **OpenAI-compatible endpoint** at `/v1`
2. We point the OpenAI client to our local Ollama server instead of OpenAI's cloud

This demonstrates the power of standardized APIs—we can switch LLM providers with minimal code changes.

In [21]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

**How this works:**
- **base_url**: Points to the local Ollama server running on `localhost:11434`
- **api_key**: For local Ollama, this is just a placeholder string (e.g., 'ollama') — no authentication needed
- Compare this to OpenAI: `OpenAI(api_key="sk-...") # cloud endpoint`

The `openai` object is now configured to send chat requests to your local Ollama instance instead of OpenAI's cloud servers.

### Understanding the Message Format

LLM conversations use a **role-based message structure**:

```
messages = [
    {"role": "system", "content": "..."},    # Sets AI behavior/expertise
    {"role": "user", "content": "..."},      # User input/question
    {"role": "assistant", "content": "..."}, # AI response (only in history)
    ...
]
```

**Roles:**
- **system**: Instructions for the AI (background, tone, constraints)
- **user**: Messages from the user
- **assistant**: Previous AI responses (builds conversational history)

The model reads the entire message list as context. Adding past exchanges helps the model understand conversation flow and maintain consistency.

In [15]:
message = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "how many letters will be in your response to my question"}
    ]

In [ ]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

### Experiment 1: No Memory / Fresh Context

Below, we ask a new question **without including the previous conversation** in the message history. The model has no context about what was asked before.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

### Experiment 2: Creating an "Illusion of Memory"

Now we ask the same question **but include prior conversation history**:

1. **User says**: "Hi! I'm Ed!" (introduction)
2. **Assistant responds**: "Hi Ed! How can I assist you today?" (our injected response)
3. **User asks**: "What's my name?" (same question as before, but now with context)

By including the full history, the model can "remember" that we introduced ourselves, creating the illusion that it recalls past interactions. In reality, **the model is stateless**—it just uses the entire message history as context for each request.

This is a fundamental pattern in LLM applications:
- **No history** → No context → Can't answer context-dependent questions
- **Full history** → Full context → Appears to have memory

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [19]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

InternalServerError: Error code: 500 - {'error': {'message': 'llama runner process has terminated: %!w(<nil>)', 'type': 'api_error', 'param': None, 'code': None}}

### Key Takeaways

✅ **Ollama provides a local, OpenAI-compatible interface** → Easy provider switching  
✅ **LLMs are stateless** → They process message history, not stored memory  
✅ **Conversation "memory" = message history** → Longer history = more context  
✅ **All chat frameworks work this way** → Claude, Gemini, Llama, etc. use the same pattern

**Practical implications:**
- For longer conversations, message history grows (costs more tokens)
- You decide what context to keep/summarize/discard
- Prompting techniques (system role, examples) matter more than "training"
- This foundation powers chatbots, Q&A systems, and multi-turn applications